In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, Image
import io
from PIL import Image as PILImage

# 1. Load Model
model_path = r'..\trainer.yml'
recognizer = cv2.face.LBPHFaceRecognizer_create()
try:
    recognizer.read(model_path)
    print(f"Model loaded from {model_path}")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please run Training.ipynb first to generate the trainer.yml file.")

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# ID to Name Mapping (Shared)
id_to_name = {
    1: "Besheer",
    2: "Ashraf",
    3: "Seif",
    4: "Sallam",
    5: "Roger",
    6: "Omar"
}

# 2. UI Elements
uploader = widgets.FileUpload(accept='image/*', multiple=False)
out_image = widgets.Output()
out_results = widgets.Output()

def process_image(change):
    out_image.clear_output()
    out_results.clear_output()
    
    if not uploader.value:
        return
        
    # Get uploaded file (ipywidgets 8.x tuple support)
    if isinstance(uploader.value, tuple):
        uploaded_file = uploader.value[0]
    else:
        uploaded_file = next(iter(uploader.value.values()))
        
    content = uploaded_file['content']
    if isinstance(content, memoryview):
        content = content.tobytes()
    
    # Convert to standard image format
    image = PILImage.open(io.BytesIO(content))
    img_np = np.array(image)
    
    # 1. Convert to Gray
    if len(img_np.shape) == 3:
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    else:
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_GRAY2BGR)
        gray = img_np
        
    # 2. Global Histogram Equalization
    gray_eq = cv2.equalizeHist(gray)
        
    # 3. Detect Faces
    faces = face_cascade.detectMultiScale(gray_eq, scaleFactor=1.1, minNeighbors=8, minSize=(60, 60))
    
    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    
    # --- STEP 1: PREDICT ALL ---
    raw_predictions = []
    
    for i, (x, y, w, h) in enumerate(faces):
        face_roi = gray_eq[y:y+h, x:x+w]
        
        # Preprocessing Chain
        face_resized = cv2.resize(face_roi, (200, 200))
        face_smooth = cv2.bilateralFilter(face_resized, 5, 75, 75)
        face_final = clahe.apply(face_smooth)
        
        # Predict
        predicted_id, conf = recognizer.predict(face_final)
        name = id_to_name.get(predicted_id, "Unknown")
        
        raw_predictions.append({
            'id': i,
            'bbox': (x, y, w, h),
            'predicted_name': name,
            'confidence': conf,
            'roi': face_final,
            'original_roi': img_np[y:y+h, x:x+w] # For UI display
        })
        
    # --- STEP 2: DEDUPLICATE (Validation) ---
    # Rule: The same person cannot be in the photo twice.
    # Logic: If duplicate names exist, keep the one with lowest confidence (best match).
    
    # Sort by confidence ascending (lower is better for LBPH)
    raw_predictions.sort(key=lambda x: x['confidence'])
    
    predictions = []
    seen_names = set()
    
    # Re-sort by original index after processing to keep spatial order logic if needed? 
    # Actually, sorting by confidence helps us pick the best easily. We can re-sort by ID later if we want stability.
    
    for pred in raw_predictions:
        name = pred['predicted_name']
        
        if name == "Unknown":
            # Unknowns are always allowed
            predictions.append(pred)
        elif name in seen_names:
            # Duplicate detected!
            # Since we sorted by best confidence, this is a worse match.
            # Demote to Unknown (or mark as Duplicate)
            pred['predicted_name'] = "Unknown" # Force to Unknown
            predictions.append(pred)
        else:
            # First time seeing this name (best match)
            seen_names.add(name)
            predictions.append(pred)
            
    # Restore original order (top-to-bottom usually)
    predictions.sort(key=lambda x: x['id'])
    
    # --- STEP 3: DRAW ---
    with out_image:
        img_display = img_np.copy()
        
        for pred in predictions:
            x, y, w, h = pred['bbox']
            name = pred['predicted_name']
            
            # Draw
            cv2.rectangle(img_display, (x, y), (x+w, y+h), (0, 255, 0), 2)
            cv2.putText(img_display, f"{name}", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
            
        display(PILImage.fromarray(img_display))
        print(f"Detected {len(faces)} faces.")

    # 3. Generate Verification UI    with out_results:
        if not predictions:
            print("No faces detected to verify.")
            return
            
        print("--- Verification ---")
        
        # Container for verification rows
        rows = []
        user_inputs = []
        
        for pred in predictions:
            # Face thumbnail
            x, y, w, h = pred['bbox']
            roi_color = img_np[y:y+h, x:x+w]
            thumb = widgets.Image(value=cv2.imencode('.png', cv2.cvtColor(roi_color, cv2.COLOR_RGB2BGR))[1].tobytes(), format='png', width=100)
            
            # Info text
            info = widgets.HTML(f"<b>Detected:</b> {pred['predicted_name']}<br><b>Conf:</b> {pred['confidence']:.2f}")
            
            # Actual Label Input
            # Using dropdown for convenience, plus 'Other'
            options = list(id_to_name.values()) + ["Unknown", "Other"]
            
            # Try to pre-select if recognized
            default_val = pred['predicted_name'] if pred['predicted_name'] in options else "Other"
            
            dropdown = widgets.Dropdown(options=options, value=default_val, description='Actual:')
            user_inputs.append({'pred': pred, 'input': dropdown})
            
            rows.append(widgets.HBox([thumb, info, dropdown]))
            
        # Calculate Button
        btn_calc = widgets.Button(description="Calculate Accuracy", button_style='success')
        lbl_acc = widgets.Label(value="")
        
        def on_calc_click(b):
            correct = 0
            total = len(user_inputs)
            
            for item in user_inputs:
                actual = item['input'].value
                predicted = item['pred']['predicted_name']
                
                # Normalize text (strip whitespace)
                actual = actual.strip()
                predicted = predicted.strip()
                
                if actual == predicted:
                    correct += 1
            
            acc = (correct / total) * 100 if total > 0 else 0
            lbl_acc.value = f"Accuracy: {acc:.2f}% ({correct}/{total})"
            
        btn_calc.on_click(on_calc_click)
        
        display(widgets.VBox(rows + [widgets.HBox([btn_calc, lbl_acc])]))

uploader.observe(process_image, names='value')

display(widgets.VBox([
    widgets.HTML("<h2>Face Recognition Tester</h2>"),
    uploader,
    out_image,
    out_results
]))


Model loaded from ..\trainer.yml
